In [22]:
import torch
from torch import nn
from d2l import torch as d2l

In [23]:
# RNN Scratch: Parameter Initialization

class RNNScratch(d2l.Module):
    
    num_inputs: int
    num_hiddens: int
    sigma: float

    def __init__(
        self,
        num_inputs: int,
        num_hiddens: int,
        sigma: float = 1e-2
    ) -> None:
        super().__init__()

        self.save_hyperparameters()
        
        # W_xh: [D, H]
        self.W_xh = nn.Parameter(
            torch.randn(
                num_inputs,
                num_hiddens,
            ) * sigma
        )
        
        # W_hh: [H, H]
        self.W_hh = nn.Parameter(
            torch.randn(
                num_hiddens,
                num_hiddens,
            ) * sigma
        )
    
        # b_h: [H]
        self.b_h = nn.Parameter(
            torch.zeros(
                num_hiddens
            )
        )

In [24]:
# Recurrent Forward Computation

def rnn_scratch_forward(
    self: RNNScratch,
    inputs: torch.Tensor,
    state: torch.Tensor | tuple[torch.Tensor] | None = None,
) -> tuple[list[torch.Tensor], torch.Tensor]:

    if state is None:
        hidden_state = torch.zeros(
            (
                inputs.shape[1],
                self.num_hiddens, 
            ),
            device=inputs.device,
        )
        
        
    elif isinstance(
        state,
        tuple,
    ): 
        hidden_state = state[0]
        
    else:
        hidden_state = state


    outputs: list[torch.Tensor] = []
    
    # inputs shape:
    # [num_steps, batch_size, num_inputs]
    for X in inputs:

        # X shape:
        # [batch_size, num_inputs]
        hidden_state = torch.tanh(
            X @ self.W_xh
            + hidden_state @ self.W_hh
            + self.b_h
        )

        outputs.append(
            hidden_state
        )
        
    return outputs, hidden_state


setattr(
    RNNScratch,
    "forward",
    rnn_scratch_forward,
)

In [25]:
# RNN Forward Test

num_steps = 100
batch_size = 2
num_inputs = 16
num_hiddens = 32

rnn = RNNScratch(
    num_inputs=num_inputs,
    num_hiddens=num_hiddens,
)

X = torch.ones(
    (
        num_steps,
        batch_size,
        num_inputs,
    )
)

outputs, state = rnn(
    X
)

print(
    "Input shape:",
    tuple(X.shape),
)

print(
    "Outputs length:",
    len(outputs),
)

print(
    "First output shape:",
    tuple(outputs[0].shape),
)

print(
    "Final state shape:",
    tuple(state.shape),
)

Input shape: (100, 2, 16)
Outputs length: 100
First output shape: (2, 32)
Final state shape: (2, 32)


In [26]:
# RNN Shape & Parameter Check

def check_len(
    values: list[torch.Tensor],
    expected_length: int,
) -> None:

    assert len(values) == expected_length, (
        f"list length {len(values)} "
        f"!= expected length {expected_length}"
    )


def check_shape(
    tensor: torch.Tensor,
    expected_shape: tuple[int, ...],
) -> None:

    actual_shape = tuple(
        tensor.shape
    )

    assert actual_shape == expected_shape, (
        f"tensor shape {actual_shape} "
        f"!= expected shape {expected_shape}"
    )


check_len(
    outputs,
    num_steps,
)

check_shape(
    outputs[0],
    (
        batch_size,
        num_hiddens,
    ),
)

check_shape(
    state,
    (
        batch_size,
        num_hiddens,
    ),
)

parameter_count = sum(
    parameter.numel()
    for parameter in rnn.parameters()
)

print(
    "Parameter count:",
    parameter_count,
)

assert parameter_count == 1568

Parameter count: 1568
